In [1]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

In [2]:
box = Box((0, 0, 0), (30, 6, 10))
box.faces.name = "outer"
cyl = sum([Cylinder((5 + (10 * i), 0, 5), Y, 2.5, 8) for i in range(3)])
cyl.faces.name = "cyl"

geo = box - cyl
geo.faces.Min(X).name = "fix"
geo.faces.Max(X).name = "force"

cylboxedges = geo.faces["outer"].edges * geo.faces["cyl"].edges
cylboxedges.name = "cylbox"
geo = geo.MakeChamfer(cylboxedges, 0.3)

mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=1)).Curve(3)

ea = {"euler_angles": (30, -30, 0)}
Draw(mesh, **ea)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (…

BaseWebGuiScene

In [ ]:
# Material Parameters
E = 21e+3
nu = 0.35
mu = E / 2 / (1 + nu)
lam = E * nu / ((1 + nu) * (1 - 2 * nu))

# Initialize FE Space
fes = VectorH1(mesh, order=2, dirichlet="fix")

# Initialize symbolic u as trial function 
# for constructing symbolic term (such as NeoHooke)
u = fes.TrialFunction()

# Components for Strain Energy
I = Id(mesh.dim)
F = I + Grad(u)
C = F.trans * F
E = 0.5 * (C - I)

def Pow(a, b):
    return a**b  # exp (log(a)*b)

def NeoHooke(C):
    return 0.5 * mu * (Trace(C - I) + 2 * mu / lam * Pow(Det(C), -lam / 2 / mu) - 1)

# Constant Loading
# force = CoefficientFunction((0, 10, 0))

# Gravity Loading
rho = 1100e-6
g = 9.81e+3
fgrav = -(rho * g)
force = CoefficientFunction((0, fgrav, 0))

# Load Factor
factor = Parameter(0)

# Stiffness Matrices
a = BilinearForm(fes, symmetric=False)
a += Variation(NeoHooke(C).Compile() * dx)
a += Variation((-factor * InnerProduct(force, u)).Compile() * dx)  # apply entire volume
# a += Variation((-factor * InnerProduct(force, u)).Compile() * ds("force"))  # apply on surface

# Initialize solution u in a format of grid function
u = GridFunction(fes)
u.vec[:] = 0  # Define an initial guess (zeros)

# Allocate the memory for calculated components in NR
res = u.vec.CreateVector()  # residual
w = u.vec.CreateVector()    # incremental displacement solution (from NR)

In [4]:
# Define solving parameters
n_loadstep = 10
n_nr = 5

# Solve with load stepping
for i_ls in range(n_loadstep):

    print(f"Load Step: {i_ls}/{n_loadstep}")
    factor.Set(i_ls/n_loadstep)

    for i_nr in range(n_nr):
        print(f"\tNewton iteration: {i_nr + 1}/{n_nr}")
        print(f"\t\tEnergy: {a.Energy(u.vec):.6f}")
        a.Apply(u.vec, res)
        a.AssembleLinearization(u.vec)
        inv = a.mat.Inverse(fes.FreeDofs())
        w.data = inv * res
        print(f"\t\tErr^2: {InnerProduct(w, res):.6e}")
        u.vec.data -= w

    Draw(u, mesh, deformation=True, scale=1, **ea)

Load Step: 0/10
	Newton iteration: 1/5
		Energy: -801197.109791
		Err^2: 4.556699e-26
	Newton iteration: 2/5
		Energy: -801197.109791
		Err^2: 4.531664e-26
	Newton iteration: 3/5
		Energy: -801197.109791
		Err^2: 4.492545e-26
	Newton iteration: 4/5
		Energy: -801197.109791
		Err^2: 4.410634e-26
	Newton iteration: 5/5
		Energy: -801197.109791
		Err^2: 4.599311e-26


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (…

Load Step: 1/10
	Newton iteration: 1/5
		Energy: -801197.109791
		Err^2: 1.236637e+03
	Newton iteration: 2/5
		Energy: -801104.669238
		Err^2: 1.429694e+03
	Newton iteration: 3/5
		Energy: -801814.423573
		Err^2: 1.658792e-01
	Newton iteration: 4/5
		Energy: -801814.506520
		Err^2: 4.210629e-09
	Newton iteration: 5/5
		Energy: -801814.506520
		Err^2: 2.694966e-20


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (…

Load Step: 2/10
	Newton iteration: 1/5
		Energy: -803047.458078
		Err^2: 1.225607e+03
	Newton iteration: 2/5
		Energy: -802966.676701
		Err^2: 1.387702e+03
	Newton iteration: 3/5
		Energy: -803655.666593
		Err^2: 1.849785e-01
	Newton iteration: 4/5
		Energy: -803655.758637
		Err^2: 2.879837e-05
	Newton iteration: 5/5
		Energy: -803655.758652
		Err^2: 1.361473e-12


...

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (…

Load Step: 8/10
	Newton iteration: 1/5
		Energy: -837247.502057
		Err^2: 8.632694e+02
	Newton iteration: 2/5
		Energy: -837452.277314
		Err^2: 4.308063e+02
	Newton iteration: 3/5
		Energy: -837666.959764
		Err^2: 2.565634e-01
	Newton iteration: 4/5
		Energy: -837667.087135
		Err^2: 7.333517e-05
	Newton iteration: 5/5
		Energy: -837667.087172
		Err^2: 3.354654e-12


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (…

Load Step: 9/10
	Newton iteration: 1/5
		Energy: -846144.084851
		Err^2: 7.918387e+02
	Newton iteration: 2/5
		Energy: -846367.899483
		Err^2: 3.220490e+02
	Newton iteration: 3/5
		Energy: -846528.495915
		Err^2: 2.068346e-01
	Newton iteration: 4/5
		Energy: -846528.598813
		Err^2: 4.263105e-05
	Newton iteration: 5/5
		Energy: -846528.598834
		Err^2: 7.439843e-13


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': (…